# Residential Monthly Bill Prediction — Experimentation

This notebook explores the provided `data/sample/training_sample_50000.parquet` input and experiments with a model for forecasting residential monthly energy use. It also creates a deterministic residential building sample in `data/silver/customer_silver.parquet` and its cutoff-featured modeling table in `data/gold/customer_gold.parquet`. These are experimental datasets; the future Airflow implementation will formalize the bronze/silver/gold pipeline and create five task-specific gold datasets.

## Working target and assumptions
- `OBJECTID` is treated as a building identifier because the input contains no customer or meter identifier. Results are therefore a residential-building bill proxy, not yet a verified customer bill.
- `normalized_building_energy_kwh` is treated as hourly kWh per record based on its name. Confirm its precise units/normalization with the dataset owner before using predictions operationally.
- The input has hourly timestamps in 2018 but no billing-cycle dates, tariff, taxes, or fees. Calendar months are used as a provisional billing period. The estimated bill is predicted monthly kWh multiplied by an explicitly configurable flat INR/kWh tariff; this is illustrative, not a tariff model.
- The input filename says 50,000, but the inspected Parquet metadata reports 419,750,000 rows. The notebook samples 1,000 residential buildings, writes their hourly rows to silver, and creates a compact gold feature table. Do not `pd.read_parquet` the complete source file.
- Forecast cutoffs represent completed days 1–7, 1–14, and 1–21. Only readings strictly before each cutoff boundary become features. The full calendar-month observed energy is the target.
- Use chronological months for tuning, validation, and final test. No model selection may use the final test months.

Silver retains the source columns for a deterministic, stratified residential-building sample while filtering invalid rows and collapsing duplicate entity/timestamp records. Gold contains one building-month-cutoff row with engineered, leakage-safe features and an observed monthly kWh target.

## 1. Imports, paths, and experiment settings

The notebook locates the named input whether the kernel starts at the repository root or `notebooks/`. Adjust the tariff and coverage threshold only with documented assumptions.

In [1]:
from pathlib import Path
import json
import warnings

import duckdb
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

INPUT_RELATIVE_PATH = Path("data") / "sample" / "training_sample_50000.parquet"
PROJECT_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root.resolve() for root in PROJECT_ROOT_CANDIDATES if (root / INPUT_RELATIVE_PATH).is_file()),
    None,
)
if PROJECT_ROOT is None:
    expected = "\n".join(str((root / INPUT_RELATIVE_PATH).resolve()) for root in PROJECT_ROOT_CANDIDATES)
    raise FileNotFoundError(f"Could not find the required sample dataset. Checked:\n{expected}")

DATA_PATH = PROJECT_ROOT / INPUT_RELATIVE_PATH
SILVER_PATH = PROJECT_ROOT / "data" / "silver" / "customer_silver.parquet"
GOLD_PATH = PROJECT_ROOT / "data" / "gold" / "customer_gold.parquet"
REBUILD_EXPERIMENT_DATASETS = False  # Set True to resample and rebuild silver/gold from the full source.
EXPERIMENT_OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "artifacts"
TARIFF_INR_PER_KWH = 10.0  # Illustrative flat rate; replace with a verified tariff before product use.
MIN_MONTH_COVERAGE = 0.95  # Exclude incomplete observed months from the target labels.
CUTOFF_DAYS = (7, 14, 21)
RANDOM_SEED = 42
MAX_TUNING_ROWS = 100_000
MAX_FINAL_FIT_ROWS = 150_000

print(f"Dataset: {DATA_PATH}")
print(f"Output directory (created at save time): {EXPERIMENT_OUTPUT_DIR}")
print(f"DuckDB {duckdb.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

Dataset: D:\GridVision\data\sample\training_sample_50000.parquet
Output directory (created at save time): D:\GridVision\notebooks\artifacts
DuckDB 1.5.6 | pandas 2.3.3 | scikit-learn 1.7.2


## 2. Build a manageable silver sample and gold feature dataset

This cell scans only the identifying/category columns to make a deterministic 1,000-building residential sample stratified by building/use type, then filters the full source to retain all hourly observations for those sampled buildings. Silver keeps the original schema and averages duplicate `(OBJECTID, timestamp)` energy rows. Gold is aggregated and feature-engineered with DuckDB, then written to the requested paths. This initial source read can take time; later notebook experiments should read the saved silver/gold files rather than rescan the source.

In [ ]:
if SILVER_PATH.is_file() and GOLD_PATH.is_file() and not REBUILD_EXPERIMENT_DATASETS:
    silver_metadata = pq.read_metadata(SILVER_PATH)
    gold_metadata = pq.read_metadata(GOLD_PATH)
    print("Using existing silver/gold experiment datasets. Set REBUILD_EXPERIMENT_DATASETS=True to regenerate them.")
    display(pd.DataFrame({
        "dataset": ["silver", "gold"],
        "path": [str(SILVER_PATH), str(GOLD_PATH)],
        "rows": [silver_metadata.num_rows, gold_metadata.num_rows],
        "size_mib": [SILVER_PATH.stat().st_size / 1024**2, GOLD_PATH.stat().st_size / 1024**2],
    }))
else:
    exec(r"""SAMPLE_BUILDING_COUNT = 1_000
DUCKDB_TEMP_DIR = PROJECT_ROOT / "data" / "duckdb_temp"
SILVER_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_PATH.parent.mkdir(parents=True, exist_ok=True)
DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

build_con = duckdb.connect()
build_con.execute("SET threads = 4")
build_con.execute("SET memory_limit = '4GB'")
build_con.execute(f"SET temp_directory = '{DUCKDB_TEMP_DIR.as_posix()}'")

building_profiles = build_con.execute('''
    SELECT
        TRY_CAST(OBJECTID AS BIGINT) AS entity_id,
        ANY_VALUE("building.type") AS building_type,
        ANY_VALUE("SpecificUseType") AS specific_use_type
    FROM read_parquet(?)
    WHERE TRIM("GeneralUseType") = 'Residential'
      AND TRY_CAST(OBJECTID AS BIGINT) IS NOT NULL
    GROUP BY TRY_CAST(OBJECTID AS BIGINT)
''', [str(DATA_PATH)]).fetchdf()
if building_profiles.empty:
    raise ValueError("No residential building IDs were found in the input dataset.")
print(f"Residential building IDs found: {len(building_profiles):,}")

sample_count = min(SAMPLE_BUILDING_COUNT, len(building_profiles))
strata = building_profiles.groupby(["building_type", "specific_use_type"], dropna=False, observed=True).size()
ideal_quota = strata / strata.sum() * sample_count
quota = np.floor(ideal_quota).astype(int).clip(upper=strata.astype(int))
remaining = sample_count - int(quota.sum())
for stratum in (ideal_quota - np.floor(ideal_quota)).sort_values(ascending=False).index:
    if remaining == 0:
        break
    if quota[stratum] < strata[stratum]:
        quota[stratum] += 1
        remaining -= 1
if remaining:
    for stratum in strata.index:
        available = int(strata[stratum] - quota[stratum])
        take = min(available, remaining)
        quota[stratum] += take
        remaining -= take
        if remaining == 0:
            break

building_profiles = building_profiles.copy()
building_profiles["_stable_hash"] = pd.util.hash_pandas_object(
    building_profiles["entity_id"].astype("uint64"), index=False
).to_numpy(dtype="uint64")
selected_frames = []
for stratum, count in quota.items():
    if count:
        mask = (building_profiles["building_type"].eq(stratum[0]) | (building_profiles["building_type"].isna() & pd.isna(stratum[0])))
        mask &= (building_profiles["specific_use_type"].eq(stratum[1]) | (building_profiles["specific_use_type"].isna() & pd.isna(stratum[1])))
        selected_frames.append(building_profiles.loc[mask].nsmallest(int(count), "_stable_hash"))
selected_buildings = pd.concat(selected_frames, ignore_index=True)[["entity_id"]]
if len(selected_buildings) != sample_count or selected_buildings["entity_id"].nunique() != sample_count:
    raise RuntimeError(f"Expected {sample_count} unique sampled buildings; got {len(selected_buildings)} rows.")
print(f"Selected {sample_count:,} buildings using a deterministic stratified sample (seed {RANDOM_SEED}).")
display(building_profiles.merge(selected_buildings, on="entity_id").groupby(["building_type", "specific_use_type"], dropna=False).size().rename("sampled_buildings").to_frame())

build_con.register("selected_buildings_df", selected_buildings)
silver_stage = SILVER_PATH.with_name(SILVER_PATH.stem + ".stage.parquet")
gold_stage = GOLD_PATH.with_name(GOLD_PATH.stem + ".stage.parquet")
for stage_path in (silver_stage, gold_stage):
    if stage_path.exists():
        stage_path.unlink()
source_sql_path = str(DATA_PATH).replace("'", "''")
silver_sql_path = str(silver_stage).replace("'", "''")

silver_copy_sql = f'''COPY (
    SELECT
        r.OBJECTID,
        ANY_VALUE(r."idf.kw") AS "idf.kw",
        ANY_VALUE(r."id.grid.coarse") AS "id.grid.coarse",
        ANY_VALUE(r."building.area.m2") AS "building.area.m2",
        ANY_VALUE(r."FootprintArea.m2") AS "FootprintArea.m2",
        ANY_VALUE(r."GeneralUseType") AS "GeneralUseType",
        ANY_VALUE(r."SpecificUseType") AS "SpecificUseType",
        ANY_VALUE(r."EffectiveYearBuilt") AS "EffectiveYearBuilt",
        ANY_VALUE(r."building.type") AS "building.type",
        ANY_VALUE(r."vintage") AS "vintage",
        ANY_VALUE(r."usetype") AS "usetype",
        ANY_VALUE(r."id.grid.finer") AS "id.grid.finer",
        ANY_VALUE(r."id.tract") AS "id.tract",
        ANY_VALUE(r."latitude") AS "latitude",
        ANY_VALUE(r."longitude") AS "longitude",
        CAST(r."timestamp" AS TIMESTAMP) AS "timestamp",
        AVG(TRY_CAST(r."normalized_building_energy_kwh" AS DOUBLE)) AS "normalized_building_energy_kwh"
    FROM read_parquet('{source_sql_path}') AS r
    INNER JOIN selected_buildings_df AS selected
        ON TRY_CAST(r."OBJECTID" AS BIGINT) = selected.entity_id
    WHERE TRIM(r."GeneralUseType") = 'Residential'
      AND TRY_CAST(r."timestamp" AS TIMESTAMP) IS NOT NULL
      AND TRY_CAST(r."normalized_building_energy_kwh" AS DOUBLE) >= 0
    GROUP BY r."OBJECTID", CAST(r."timestamp" AS TIMESTAMP)
    ORDER BY "timestamp", r."OBJECTID"
) TO '{silver_sql_path}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)'''
build_con.execute(silver_copy_sql)
silver_check = pq.read_metadata(silver_stage)
if silver_check.num_rows == 0:
    raise RuntimeError("Silver dataset was written but contains no records.")
silver_stage.replace(SILVER_PATH)
print(f"Silver saved: {SILVER_PATH} | rows={silver_check.num_rows:,} | row groups={silver_check.num_row_groups:,}")

gold_sql_path = str(gold_stage).replace("'", "''")
gold_copy_sql = f'''COPY (
WITH hourly AS (
    SELECT * FROM read_parquet('{str(SILVER_PATH).replace("'", "''")}')
), monthly AS (
    SELECT
        CAST("OBJECTID" AS BIGINT) AS entity_id,
        DATE_TRUNC('month', "timestamp")::DATE AS month_start,
        ANY_VALUE("idf.kw") AS idf_kw,
        ANY_VALUE("id.grid.coarse") AS id_grid_coarse,
        ANY_VALUE("building.area.m2") AS building_area_m2,
        ANY_VALUE("FootprintArea.m2") AS footprint_area_m2,
        ANY_VALUE("SpecificUseType") AS specific_use_type,
        ANY_VALUE("EffectiveYearBuilt") AS effective_year_built,
        ANY_VALUE("building.type") AS building_type,
        ANY_VALUE("vintage") AS vintage,
        ANY_VALUE("usetype") AS usetype,
        ANY_VALUE("id.grid.finer") AS id_grid_finer,
        ANY_VALUE("id.tract") AS id_tract,
        ANY_VALUE("latitude") AS latitude,
        ANY_VALUE("longitude") AS longitude,
        SUM("normalized_building_energy_kwh") AS monthly_kwh,
        COUNT(*) AS observed_hours,
        24 * DAY(LAST_DAY(ANY_VALUE("timestamp"))) AS expected_hours,
        SUM("normalized_building_energy_kwh") FILTER (WHERE "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 7 DAY) AS energy_days_1_7_kwh,
        COUNT(*) FILTER (WHERE "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 7 DAY) AS hours_days_1_7,
        SUM("normalized_building_energy_kwh") FILTER (WHERE "timestamp" >= DATE_TRUNC('month', "timestamp") + INTERVAL 7 DAY AND "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 14 DAY) AS energy_days_8_14_kwh,
        COUNT(*) FILTER (WHERE "timestamp" >= DATE_TRUNC('month', "timestamp") + INTERVAL 7 DAY AND "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 14 DAY) AS hours_days_8_14,
        SUM("normalized_building_energy_kwh") FILTER (WHERE "timestamp" >= DATE_TRUNC('month', "timestamp") + INTERVAL 14 DAY AND "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 21 DAY) AS energy_days_15_21_kwh,
        COUNT(*) FILTER (WHERE "timestamp" >= DATE_TRUNC('month', "timestamp") + INTERVAL 14 DAY AND "timestamp" < DATE_TRUNC('month', "timestamp") + INTERVAL 21 DAY) AS hours_days_15_21,
        CAST(ANY_VALUE("timestamp") AS DATE) AS any_timestamp
    FROM hourly
    GROUP BY CAST("OBJECTID" AS BIGINT), DATE_TRUNC('month', "timestamp")::DATE
), complete_months AS (
    SELECT *, observed_hours * 1.0 / expected_hours AS coverage_ratio
    FROM monthly
    WHERE observed_hours * 1.0 / expected_hours >= {MIN_MONTH_COVERAGE}
), lagged AS (
    SELECT *,
        LAG(monthly_kwh) OVER (PARTITION BY entity_id ORDER BY month_start) AS prior_month_candidate,
        LAG(month_start) OVER (PARTITION BY entity_id ORDER BY month_start) AS prior_month_date
    FROM complete_months
), base AS (
    SELECT *,
        CASE WHEN prior_month_date = month_start - INTERVAL 1 MONTH THEN prior_month_candidate END AS prior_month_kwh,
        DAY(LAST_DAY(month_start)) AS days_in_month,
        MONTH(month_start) AS month_of_year,
        SIN(2 * PI() * MONTH(month_start) / 12.0) AS month_sin,
        COS(2 * PI() * MONTH(month_start) / 12.0) AS month_cos
    FROM lagged
), cutoffs AS (
    SELECT *, 7 AS cutoff_day,
        energy_days_1_7_kwh AS cumulative_energy_kwh,
        energy_days_1_7_kwh AS recent_7d_energy_kwh,
        hours_days_1_7 AS observed_hours_to_cutoff,
        hours_days_1_7 AS recent_7d_hours
    FROM base
    UNION ALL
    SELECT *, 14,
        energy_days_1_7_kwh + energy_days_8_14_kwh,
        energy_days_8_14_kwh,
        hours_days_1_7 + hours_days_8_14,
        hours_days_8_14
    FROM base
    UNION ALL
    SELECT *, 21,
        energy_days_1_7_kwh + energy_days_8_14_kwh + energy_days_15_21_kwh,
        energy_days_15_21_kwh,
        hours_days_1_7 + hours_days_8_14 + hours_days_15_21,
        hours_days_15_21
    FROM base
)
SELECT
    entity_id, month_start, idf_kw, id_grid_coarse, building_area_m2, footprint_area_m2,
    specific_use_type, effective_year_built, building_type, vintage, usetype,
    id_grid_finer, id_tract, latitude, longitude,
    coverage_ratio, observed_hours, expected_hours, cutoff_day, days_in_month, month_of_year, month_sin, month_cos,
    prior_month_kwh,
    prior_month_kwh / NULLIF(building_area_m2, 0) AS prior_month_kwh_per_m2,
    observed_hours_to_cutoff, cumulative_energy_kwh, recent_7d_energy_kwh,
    recent_7d_energy_kwh / NULLIF(recent_7d_hours, 0) AS recent_7d_mean_hourly_kwh,
    cumulative_energy_kwh / NULLIF(observed_hours_to_cutoff, 0) AS mean_hourly_energy_to_cutoff,
    cumulative_energy_kwh / NULLIF(building_area_m2, 0) AS cumulative_energy_per_m2,
    cumulative_energy_kwh / cutoff_day * days_in_month AS naive_projected_month_kwh,
    monthly_kwh AS target_month_kwh,
    monthly_kwh * {TARIFF_INR_PER_KWH} AS target_bill_inr
FROM cutoffs
) TO '{gold_sql_path}' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 100000)'''
build_con.execute(gold_copy_sql)
gold_check = pq.read_metadata(gold_stage)
if gold_check.num_rows == 0:
    raise RuntimeError("Gold dataset was written but contains no records.")
gold_stage.replace(GOLD_PATH)
build_con.close()
print(f"Gold saved: {GOLD_PATH} | rows={gold_check.num_rows:,} | row groups={gold_check.num_row_groups:,}")
print("Gold rows are building × complete month × cutoff (days 7, 14, 21).")
display(pd.DataFrame({
    "dataset": ["silver", "gold"],
    "path": [str(SILVER_PATH), str(GOLD_PATH)],
    "rows": [silver_check.num_rows, gold_check.num_rows],
    "columns": [len(pq.read_schema(SILVER_PATH)), len(pq.read_schema(GOLD_PATH))],
    "size_mib": [SILVER_PATH.stat().st_size / 1024**2, GOLD_PATH.stat().st_size / 1024**2],
}))
""")

## 3. Inspect the source Parquet schema and a bounded sample

This cell reads Parquet metadata and a small preview only. The preview is from the file's leading row groups and is useful for checking representation, not for claiming dataset-wide distributions.

In [2]:
parquet_metadata = pq.read_metadata(DATA_PATH)
parquet_schema = pq.read_schema(DATA_PATH)
print(f"Rows: {parquet_metadata.num_rows:,} | row groups: {parquet_metadata.num_row_groups:,}")
display(pd.DataFrame({"column": parquet_schema.names, "type": [str(field.type) for field in parquet_schema]}))

con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET memory_limit = '4GB'")
preview = con.execute("SELECT * FROM read_parquet(?) LIMIT 10000", [str(DATA_PATH)]).fetchdf()
display(preview.head(10))
display(pd.DataFrame({
    "dtype_in_preview": preview.dtypes.astype(str),
    "missing_in_preview": preview.isna().sum(),
    "distinct_in_preview": preview.nunique(dropna=True),
}))
if "GeneralUseType" in preview:
    display(preview["GeneralUseType"].value_counts(dropna=False).rename("preview_rows"))

Rows: 419,750,000 | row groups: 3,401


,column,type
0,OBJECTID,int64
1,idf.kw,string
2,id.grid.coarse,int64
3,building.area.m2,double
4,FootprintArea.m2,double
5,GeneralUseType,string
6,SpecificUseType,string
7,EffectiveYearBuilt,int64
8,building.type,string
9,vintage,string


,OBJECTID,idf.kw,id.grid.coarse,building.area.m2,FootprintArea.m2,GeneralUseType,SpecificUseType,EffectiveYearBuilt,building.type,vintage,usetype,id.grid.finer,id.tract,latitude,longitude,timestamp,normalized_building_energy_kwh
0,264732,MultiFamily-pre-1980,62,173.3514,166.941284,Residential,"Double, Duplex, or Two Units",1963,multi-family,before 1980,multi_family,45578.0,06037572201,33.814844,-118.190995,2018-12-26 18:00:00,1.290890
1,264939,Warehouse-90_1-2004-ASHRAE 169-2013-3B,63,3589.6560,3591.126368,Industrial,"Warehousing, Distribution, Storage",1997,warehouse,1981-2008 (2004),industrial,44914.0,06037543305,33.866706,-118.211209,2018-12-26 18:00:00,33.673006
2,265053,SingleFamily-pre-1980,62,169.6354,207.979102,Residential,Single Family Residence,1956,single-family,before 1980,single_family,50657.0,06037574400,33.805040,-118.113777,2018-12-26 18:00:00,0.787332
3,265095,SingleFamily-pre-1980,62,152.9134,228.991269,Residential,Single Family Residence,1956,single-family,before 1980,single_family,49642.0,06037574201,33.809100,-118.127263,2018-12-26 18:00:00,0.709719
4,265097,SingleFamily-pre-1980,62,126.9014,140.447519,Residential,Single Family Residence,1950,single-family,before 1980,single_family,49987.0,06037571200,33.833983,-118.129931,2018-12-26 18:00:00,0.588990
5,265139,MultiFamily-pre-1980,35,5612.4606,221.079927,Residential,Five or More Units or Apartments (Any Combinat...,1973,multi-family,before 1980,multi_family,32030.0,06037621326,33.826925,-118.389589,2018-12-26 18:00:00,41.794115
6,265179,SingleFamily-pre-1980,35,119.8410,246.632189,Residential,Single Family Residence,1954,single-family,before 1980,single_family,32364.0,06037651302,33.805946,-118.380378,2018-12-26 18:00:00,0.574839
7,265191,SingleFamily-pre-1980,35,129.5026,240.086122,Residential,Single Family Residence,1955,single-family,before 1980,single_family,32364.0,06037651302,33.808077,-118.382871,2018-12-26 18:00:00,0.621182
8,265276,SingleFamily-pre-1980,49,102.0042,146.632511,Residential,Single Family Residence,1954,single-family,before 1980,single_family,35086.0,06037650501,33.854761,-118.356078,2018-02-18 12:00:00,0.262556
9,265279,SingleFamily-pre-1980,35,167.6845,133.235140,Residential,Single Family Residence,1963,single-family,before 1980,single_family,32040.0,06037621004,33.866490,-118.400977,2018-12-26 18:00:00,0.804329


,dtype_in_preview,missing_in_preview,distinct_in_preview
OBJECTID,int64,0,253
idf.kw,object,0,15
id.grid.coarse,int64,0,8
building.area.m2,float64,0,244
FootprintArea.m2,float64,0,253
GeneralUseType,object,0,3
SpecificUseType,object,0,13
EffectiveYearBuilt,int64,0,86
building.type,object,0,9
vintage,object,0,3


GeneralUseType
Residential    9343
Commercial      398
Industrial      259
Name: preview_rows, dtype: int64

## 4. Establish the modeling grain and data-quality rules

The prediction grain is one residential `OBJECTID` × calendar month × forecast cutoff. Repeated rows for an entity/timestamp are collapsed to one hourly observation using the mean energy value, so duplicate records do not inflate monthly totals. Null timestamps, null energy, and negative energy are excluded from this experiment; zero energy remains valid. We retain only building-month labels with at least the configured fraction of expected hourly timestamps.

These are provisional rules for investigation. Review duplicate/conflicting records, timestamp timezone, missing-hour patterns, unit semantics, and whether `OBJECTID` truly represents a billable customer before treating the derived dataset as gold.

In [ ]:
monthly_sql = '''
WITH source_rows AS (
    SELECT
        TRY_CAST(OBJECTID AS BIGINT) AS entity_id,
        TRY_CAST(timestamp AS TIMESTAMP) AS observed_at,
        TRY_CAST(normalized_building_energy_kwh AS DOUBLE) AS energy_kwh,
        TRY_CAST("building.area.m2" AS DOUBLE) AS building_area_m2,
        TRY_CAST("FootprintArea.m2" AS DOUBLE) AS footprint_area_m2,
        TRY_CAST(EffectiveYearBuilt AS DOUBLE) AS effective_year_built,
        TRY_CAST(latitude AS DOUBLE) AS latitude,
        TRY_CAST(longitude AS DOUBLE) AS longitude,
        "SpecificUseType" AS specific_use_type,
        "building.type" AS building_type,
        vintage
    FROM read_parquet(?)
    WHERE TRIM(GeneralUseType) = 'Residential'
),
valid_rows AS (
    SELECT *
    FROM source_rows
    WHERE entity_id IS NOT NULL
      AND observed_at IS NOT NULL
      AND energy_kwh IS NOT NULL
      AND energy_kwh >= 0
),
hourly AS (
    SELECT
        entity_id,
        observed_at,
        AVG(energy_kwh) AS energy_kwh,
        ANY_VALUE(building_area_m2) AS building_area_m2,
        ANY_VALUE(footprint_area_m2) AS footprint_area_m2,
        ANY_VALUE(effective_year_built) AS effective_year_built,
        ANY_VALUE(latitude) AS latitude,
        ANY_VALUE(longitude) AS longitude,
        ANY_VALUE(specific_use_type) AS specific_use_type,
        ANY_VALUE(building_type) AS building_type,
        ANY_VALUE(vintage) AS vintage,
        COUNT(*) AS source_row_count
    FROM valid_rows
    GROUP BY entity_id, observed_at
),
monthly AS (
    SELECT
        entity_id,
        DATE_TRUNC('month', observed_at)::DATE AS month_start,
        ANY_VALUE(building_area_m2) AS building_area_m2,
        ANY_VALUE(footprint_area_m2) AS footprint_area_m2,
        ANY_VALUE(effective_year_built) AS effective_year_built,
        ANY_VALUE(latitude) AS latitude,
        ANY_VALUE(longitude) AS longitude,
        ANY_VALUE(specific_use_type) AS specific_use_type,
        ANY_VALUE(building_type) AS building_type,
        ANY_VALUE(vintage) AS vintage,
        SUM(energy_kwh) AS monthly_kwh,
        COUNT(*) AS unique_observed_hours,
        SUM(source_row_count) AS source_rows,
        SUM(source_row_count - 1) AS duplicate_source_rows,
        SUM(CASE WHEN observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 7 DAY THEN energy_kwh ELSE 0 END) AS energy_days_1_7_kwh,
        COUNT(*) FILTER (WHERE observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 7 DAY) AS hours_days_1_7,
        SUM(CASE WHEN observed_at >= DATE_TRUNC('month', observed_at) + INTERVAL 7 DAY AND observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 14 DAY THEN energy_kwh ELSE 0 END) AS energy_days_8_14_kwh,
        COUNT(*) FILTER (WHERE observed_at >= DATE_TRUNC('month', observed_at) + INTERVAL 7 DAY AND observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 14 DAY) AS hours_days_8_14,
        SUM(CASE WHEN observed_at >= DATE_TRUNC('month', observed_at) + INTERVAL 14 DAY AND observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 21 DAY THEN energy_kwh ELSE 0 END) AS energy_days_15_21_kwh,
        COUNT(*) FILTER (WHERE observed_at >= DATE_TRUNC('month', observed_at) + INTERVAL 14 DAY AND observed_at < DATE_TRUNC('month', observed_at) + INTERVAL 21 DAY) AS hours_days_15_21
    FROM hourly
    GROUP BY entity_id, DATE_TRUNC('month', observed_at)::DATE
)
SELECT * FROM monthly
ORDER BY month_start, entity_id
'''

monthly = con.execute(monthly_sql, [str(SILVER_PATH)]).fetchdf()
monthly["month_start"] = pd.to_datetime(monthly["month_start"])
print(f"Residential entity-month rows before completeness filtering: {len(monthly):,}")
print(f"Unique residential entities: {monthly['entity_id'].nunique():,}")
print(f"Observed time range: {monthly['month_start'].min():%Y-%m} through {monthly['month_start'].max():%Y-%m}")
print(f"Duplicate source rows collapsed: {int(monthly['duplicate_source_rows'].sum()):,}")
display(monthly.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 5. Inspect monthly coverage, target distribution, and categories

Coverage uses expected calendar hours (24 × days in month). Local timestamps may have daylight-saving irregularities, so the threshold is a reviewable starting point rather than an assertion that every month must have exactly the same number of intervals. The target is observed monthly energy after duplicate collapse and valid-reading filters.

In [ ]:
monthly["days_in_month"] = (monthly["month_start"] + pd.offsets.MonthEnd(0)).dt.day
monthly["expected_hours"] = monthly["days_in_month"] * 24
monthly["coverage_ratio"] = monthly["unique_observed_hours"] / monthly["expected_hours"]
print("Coverage ratio summary:")
display(monthly["coverage_ratio"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).to_frame())
coverage_by_month = monthly.groupby("month_start", observed=True).agg(
    entity_months=("entity_id", "size"),
    entities=("entity_id", "nunique"),
    median_coverage=("coverage_ratio", "median"),
    p05_coverage=("coverage_ratio", lambda values: values.quantile(0.05)),
    median_monthly_kwh=("monthly_kwh", "median"),
)
display(coverage_by_month)
display(monthly["monthly_kwh"].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).to_frame())
for column in ["specific_use_type", "building_type", "vintage"]:
    print(f"Top values for {column}:")
    display(monthly[column].value_counts(dropna=False).head(20).to_frame("entity_months"))

monthly_complete = monthly.loc[monthly["coverage_ratio"] >= MIN_MONTH_COVERAGE].copy()
retained_pct = 100 * len(monthly_complete) / max(len(monthly), 1)
print(f"Retained at coverage >= {MIN_MONTH_COVERAGE:.0%}: {len(monthly_complete):,}/{len(monthly):,} entity-months ({retained_pct:.1f}%)")
if monthly_complete.empty:
    raise ValueError("No monthly records meet the coverage threshold; inspect timestamp cadence and revise the documented rule.")

monthly_complete = monthly_complete.sort_values(["entity_id", "month_start"]).copy()

## 6. Load the cutoff-safe engineered features from gold

The earlier build cell created the silver and gold files. This section reads the gold dataset rather than rebuilding features in memory. Each row is a building × complete calendar month × cutoff (day 7, 14, or 21). `entity_id` is retained for audit and grouped analysis but deliberately excluded from model features. Categorical building descriptors and static size/location attributes are candidate predictors.

In [ ]:
examples = pd.read_parquet(GOLD_PATH)
examples["month_start"] = pd.to_datetime(examples["month_start"])
CATEGORICAL_FEATURES = ["specific_use_type", "building_type", "vintage"]
NUMERIC_FEATURES = [
    "building_area_m2", "footprint_area_m2", "effective_year_built", "latitude", "longitude",
    "prior_month_kwh", "prior_month_kwh_per_m2",
    "cutoff_day", "days_in_month", "observed_hours_to_cutoff", "cumulative_energy_kwh",
    "recent_7d_energy_kwh", "recent_7d_mean_hourly_kwh", "mean_hourly_energy_to_cutoff",
    "cumulative_energy_per_m2", "naive_projected_month_kwh", "month_of_year", "month_sin", "month_cos",
]
MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
for column in CATEGORICAL_FEATURES:
    examples[column] = examples[column].astype("category")
for column in NUMERIC_FEATURES:
    examples[column] = pd.to_numeric(examples[column], errors="coerce").astype("float32")
examples["target_month_kwh"] = examples["target_month_kwh"].astype("float32")
examples["target_bill_inr"] = examples["target_bill_inr"].astype("float32")

print(f"Loaded gold dataset: {GOLD_PATH}")
print(f"Model examples: {len(examples):,} | input features: {len(MODEL_FEATURES)}")
print(f"Cutoff examples: {examples['cutoff_day'].value_counts().sort_index().to_dict()}")
display(examples[MODEL_FEATURES + ["target_month_kwh", "target_bill_inr"]].head())
display(examples.groupby("cutoff_day", observed=True)[["target_month_kwh", "naive_projected_month_kwh"]].agg(["count", "median", "mean"]))

## 7. Explore seasonality, target skew, and baseline quality

The simple baseline extrapolates observed average daily kWh through the calendar month. It is a useful benchmark, not a trained model. Metrics are reported in kWh and converted to INR only by the configured flat rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
examples.groupby("month_start", observed=True)["target_month_kwh"].median().plot(ax=axes[0], marker="o")
axes[0].set_title("Median residential building-month kWh")
axes[0].set_ylabel("kWh per calendar month")
examples["target_month_kwh"].clip(upper=examples["target_month_kwh"].quantile(0.99)).hist(bins=50, ax=axes[1])
axes[1].set_title("Monthly target (display capped at p99)")
axes[1].set_xlabel("kWh; display only capped at p99")
plt.tight_layout()
plt.show()

def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return {
        "MAE_kWh": mean_absolute_error(y_true, y_pred),
        "RMSE_kWh": np.sqrt(mean_squared_error(y_true, y_pred)),
        "WAPE_pct": 100 * np.abs(y_true - y_pred).sum() / max(np.abs(y_true).sum(), 1e-9),
        "R2": r2_score(y_true, y_pred),
        "MAE_INR_at_configured_tariff": mean_absolute_error(y_true, y_pred) * TARIFF_INR_PER_KWH,
    }

examples.groupby("cutoff_day", observed=True).apply(
    lambda group: pd.Series(regression_metrics(group["target_month_kwh"], group["naive_projected_month_kwh"])),
    include_groups=False,
).rename_axis("cutoff_day").reset_index()

## 8. Define chronological tuning, validation, and final test periods

For the inspected 2018 data, January–August are used for expanding-window hyperparameter tuning, September–October for validation, and November–December for the untouched final test. If coverage filtering removes a split period, stop and revisit data coverage rather than silently mixing time periods. Since only one year is available, evidence about year-to-year generalization will remain limited.

In [ ]:
TRAIN_END = pd.Timestamp("2018-09-01")
VALIDATION_START = pd.Timestamp("2018-09-01")
TEST_START = pd.Timestamp("2018-11-01")
tuning_data = examples.loc[examples["month_start"] < TRAIN_END].copy()
validation_data = examples.loc[
    (examples["month_start"] >= VALIDATION_START) & (examples["month_start"] < TEST_START)
].copy()
final_test_data = examples.loc[examples["month_start"] >= TEST_START].copy()
if tuning_data.empty or validation_data.empty or final_test_data.empty:
    available = sorted(examples["month_start"].dt.strftime("%Y-%m").unique())
    raise ValueError(f"Required chronological periods are empty. Available months: {available}")
print(f"Tuning rows: {len(tuning_data):,} | validation rows: {len(validation_data):,} | final test rows: {len(final_test_data):,}")
print("Tuning months:", sorted(tuning_data["month_start"].dt.strftime("%Y-%m").unique()))
print("Validation months:", sorted(validation_data["month_start"].dt.strftime("%Y-%m").unique()))
print("Final test months:", sorted(final_test_data["month_start"].dt.strftime("%Y-%m").unique()))

def sample_by_month(frame, max_rows, seed=RANDOM_SEED):
    if len(frame) <= max_rows:
        return frame.copy().reset_index(drop=True)
    month_count = frame["month_start"].nunique()
    per_month = max(max_rows // month_count, 1)
    sampled = [
        group.sample(n=min(len(group), per_month), random_state=seed)
        for _, group in frame.groupby("month_start", observed=True, sort=True)
    ]
    return pd.concat(sampled, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

tuning_sample = sample_by_month(tuning_data, MAX_TUNING_ROWS)
print(f"Rows used for tuning: {len(tuning_sample):,} (deterministic, sampled evenly by month when needed)")

## 9. Tune a gradient-boosting model with expanding-window CV

Validation folds preserve chronology: each fold trains on earlier months and validates on a later month. We cap and deterministically sample the tuning set to make experimentation practical; final test evaluation below uses all retained examples in the test period. One-hot encoding is fitted inside each fold, and unseen categories are handled without leaking validation information.

In [ ]:
tuning_months = sorted(tuning_sample["month_start"].dt.to_period("M").unique())
if len(tuning_months) < 5:
    raise ValueError(f"Need at least five tuning months for expanding-window CV; found {len(tuning_months)}.")
month_periods = tuning_sample["month_start"].dt.to_period("M")
cv_splits = []
for validation_position in range(4, len(tuning_months)):
    earlier_months = tuning_months[:validation_position]
    later_month = tuning_months[validation_position]
    train_indices = np.flatnonzero(month_periods.isin(earlier_months).to_numpy())
    validation_indices = np.flatnonzero((month_periods == later_month).to_numpy())
    if len(train_indices) and len(validation_indices):
        cv_splits.append((train_indices, validation_indices))
print(f"Expanding-window CV folds: {len(cv_splits)}")
if not cv_splits:
    raise ValueError("No non-empty chronological CV splits could be created.")

numeric_transformer = Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, NUMERIC_FEATURES),
    ("categorical", categorical_transformer, CATEGORICAL_FEATURES),
], remainder="drop")
experiment_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingRegressor(
        loss="absolute_error", random_state=RANDOM_SEED, early_stopping=False
    )),
])
parameter_distributions = {
    "model__max_iter": [100, 200],
    "model__learning_rate": [0.03, 0.06, 0.1],
    "model__max_leaf_nodes": [15, 31],
    "model__min_samples_leaf": [20, 50],
    "model__l2_regularization": [0.0, 1.0],
}
search = RandomizedSearchCV(
    estimator=experiment_pipeline,
    param_distributions=parameter_distributions,
    n_iter=6,
    scoring="neg_mean_absolute_error",
    cv=cv_splits,
    random_state=RANDOM_SEED,
    n_jobs=1,
    refit=True,
    verbose=1,
    return_train_score=False,
)
X_tuning = tuning_sample[MODEL_FEATURES]
y_tuning = tuning_sample["target_month_kwh"]
search.fit(X_tuning, y_tuning)
best_pipeline = search.best_estimator_
print("Best CV MAE (kWh):", -search.best_score_)
print("Best parameters:", search.best_params_)
display(pd.DataFrame(search.cv_results_).sort_values("rank_test_score").head(10)[
    ["rank_test_score", "mean_test_score", "std_test_score", "params"]
])

## 10. Compare validation performance and then refit

Compare the tuned model with the naive projection on September–October. After recording validation results, refit the chosen hyperparameters using January–October only. The November–December test data remains untouched until the next section.

In [ ]:
X_validation = validation_data[MODEL_FEATURES]
y_validation = validation_data["target_month_kwh"]
validation_model_predictions = best_pipeline.predict(X_validation)
validation_baseline_predictions = validation_data["naive_projected_month_kwh"].to_numpy()
validation_results = pd.DataFrame([
    {"model": "naive_daily_extrapolation", **regression_metrics(y_validation, validation_baseline_predictions)},
    {"model": "tuned_hist_gradient_boosting", **regression_metrics(y_validation, validation_model_predictions)},
])
display(validation_results)
validation_winner = validation_results.loc[validation_results["MAE_kWh"].idxmin(), "model"]
model_wins_validation = validation_winner == "tuned_hist_gradient_boosting"
print(f"Validation winner: {validation_winner}. Tuned model promotion recommended: {model_wins_validation}.")
validation_by_cutoff = []
for cutoff_day, group in validation_data.groupby("cutoff_day", observed=True):
    positions = group.index
    predictions = best_pipeline.predict(group[MODEL_FEATURES])
    validation_by_cutoff.append({
        "cutoff_day": int(cutoff_day),
        **regression_metrics(group["target_month_kwh"], predictions),
    })
display(pd.DataFrame(validation_by_cutoff))

fit_period = examples.loc[examples["month_start"] < TEST_START].copy()
final_fit_sample = sample_by_month(fit_period, MAX_FINAL_FIT_ROWS)
final_model = best_pipeline
final_model.fit(final_fit_sample[MODEL_FEATURES], final_fit_sample["target_month_kwh"])
print(f"Final refit rows: {len(final_fit_sample):,}; includes tuning + validation months only.")

## 11. Evaluate once on the held-out final test period

Report the model and naive baseline on November–December overall and by cutoff. Treat these as a limited one-year temporal test, not proof of future-year or unseen-customer performance. Inspect systematic errors and high-usage outliers before adopting the model.

In [ ]:
X_test = final_test_data[MODEL_FEATURES]
y_test = final_test_data["target_month_kwh"]
test_predictions = np.maximum(final_model.predict(X_test), 0)
test_baseline_predictions = np.maximum(final_test_data["naive_projected_month_kwh"].to_numpy(), 0)
test_results = pd.DataFrame([
    {"model": "naive_daily_extrapolation", **regression_metrics(y_test, test_baseline_predictions)},
    {"model": "tuned_hist_gradient_boosting", **regression_metrics(y_test, test_predictions)},
])
display(test_results)

test_scored = final_test_data[["entity_id", "month_start", "cutoff_day", "target_month_kwh", "target_bill_inr"]].copy()
test_scored["predicted_month_kwh"] = test_predictions
test_scored["predicted_bill_inr"] = test_scored["predicted_month_kwh"] * TARIFF_INR_PER_KWH
test_scored["absolute_error_kwh"] = np.abs(test_scored["target_month_kwh"] - test_scored["predicted_month_kwh"])
display(test_scored.groupby("cutoff_day", observed=True).apply(
    lambda group: pd.Series(regression_metrics(group["target_month_kwh"], group["predicted_month_kwh"])),
    include_groups=False,
).rename_axis("cutoff_day").reset_index())
display(test_scored.sort_values("absolute_error_kwh", ascending=False).head(20))

## 12. Inspect permutation importance on a bounded test sample

Permutation importance is diagnostic and can be noisy when features are correlated. It is computed on a deterministic sample of the final test period to limit runtime.

In [ ]:
importance_sample = sample_by_month(final_test_data, 10_000)
importance = permutation_importance(
    final_model,
    importance_sample[MODEL_FEATURES],
    importance_sample["target_month_kwh"],
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=RANDOM_SEED,
    n_jobs=1,
)
importance_table = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "mae_increase_kwh_mean": importance.importances_mean,
    "mae_increase_kwh_std": importance.importances_std,
}).sort_values("mae_increase_kwh_mean", ascending=False)
display(importance_table)

## 13. Save the best tuned experiment artifact

Save the fitted preprocessing + estimator pipeline and the assumptions needed to interpret it. This is a local experimental artifact, not a production model registry. Later production training should move reusable logic into Python modules and use MLflow for versioning, parameters, metrics, and artifact tracking.

In [ ]:
EXPERIMENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = EXPERIMENT_OUTPUT_DIR / "customer_bill_prediction_experiment.joblib"
test_model_row = test_results.loc[test_results["model"] == "tuned_hist_gradient_boosting"].iloc[0].to_dict()
test_baseline_row = test_results.loc[test_results["model"] == "naive_daily_extrapolation"].iloc[0].to_dict()
artifact = {
    "pipeline": final_model,
    "metadata": {
        "experiment": "residential_monthly_bill_prediction",
        "input_path_at_training": str(DATA_PATH),
        "target": "calendar_month_observed_energy_kwh",
        "bill_formula": "predicted_month_kwh * tariff_inr_per_kwh",
        "tariff_inr_per_kwh_assumption": TARIFF_INR_PER_KWH,
        "entity_key": "OBJECTID (provisional building proxy; not verified customer/meter ID)",
        "cutoff_days": list(CUTOFF_DAYS),
        "minimum_month_coverage": MIN_MONTH_COVERAGE,
        "feature_names": MODEL_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "best_hyperparameters": search.best_params_,
        "validation_metrics": validation_results.to_dict(orient="records"),
        "final_test_model_metrics": test_model_row,
        "final_test_baseline_metrics": test_baseline_row,
        "validation_winner": validation_winner,
        "tuned_model_wins_validation": bool(model_wins_validation),
        "tuned_model_wins_final_test": bool(test_model_row["MAE_kWh"] < test_baseline_row["MAE_kWh"]),
        "train_validation_period": "2018-01 through 2018-10",
        "final_test_period": "2018-11 through 2018-12",
        "sampled_final_fit_rows": int(len(final_fit_sample)),
        "versions": {"python_packages": {"pandas": pd.__version__, "scikit_learn": sklearn.__version__, "duckdb": duckdb.__version__}},
        "limitations": [
            "Calendar-month periods substitute for unavailable billing-cycle dates.",
            "OBJECTID is a provisional building proxy, not a verified customer or meter identifier.",
            "Energy unit interpretation and flat tariff require domain confirmation.",
            "Only one calendar year is available for temporal evaluation.",
            "Tuning/final fitting may use deterministic row samples when configured row limits are exceeded.",
        ],
    },
}
joblib.dump(artifact, MODEL_PATH, compress=3)
print(f"Saved experimental model artifact: {MODEL_PATH}")
print(f"File size: {MODEL_PATH.stat().st_size / (1024 ** 2):.2f} MiB")

## 14. Conclusions and next decisions

Review the validation/test tables and feature importance before accepting the experiment. Specifically verify the energy units, customer/building identity, local timestamp timezone, actual invoice cycle, tariff schedule, duplicate policy, and complete-month coverage behavior with the data owner. A useful next experiment is a household/building holdout split in addition to chronological evaluation.

Once the analytical rules are agreed, implement reproducible bronze → silver → five task-specific gold datasets and their Airflow orchestration separately. Then move approved feature and training logic into importable Python modules, add tests, and track training runs/models/metrics with MLflow. This notebook intentionally does not implement Airflow or MLflow.